In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
import pandas as pd
import os
import uuid
import shutil

In [ ]:
import logging

# quiet down Kedro loggers
for name in [
    "kedro",
    "kedro.framework",
    "kedro.runner",
    "kedro.io",
    "kedro.pipeline",
    "kedro.extras",
]:
    logging.getLogger(name).setLevel(logging.WARNING)

# (optional) quiet root logger too
logging.getLogger().setLevel(logging.WARNING)

In [ ]:
# Create workspace/results_v1 folder structure
workspace_dir = Path("../workspace/results_v1")
workspace_dir.mkdir(parents=True, exist_ok=True)
print(f"Created workspace directory: {workspace_dir}")

In [ ]:
os.environ["KEDRO_PACKAGE_NAME"] = "crispy_kedro"

# Since the notebook is in ./notebooks, set the project path to the parent directory
# os.chdir(Path.cwd().parent)

metadata = bootstrap_project(project_path=Path.cwd().parent)

In [ ]:
companies_selection = [
    # multinational megacorps
    "CP_7876088876044165226",
    "CN_9186444779649860568",
    "CN_8600108312240451561",
    "CP_1512176126791706747",
    # big greentech owners
    "CN_6660639238798673502",
    "CN_6660639238798673502",
    "CN_5719632086864744401",
    # big carbontech owners
    "CN_8600108312240451561",
    "CN_7548398708980274705",
    "CN_5252218731344992786",
    # random other owners, with 10-20 assets
    "CN_8249155112555313068",
    "CP_7671368023139165011",
    "CN_7263620466430749129",
    "CP_5133603177600074280",
    "CP_1405113695717703383",
    "CN_7237425157254272056",
    # random other owners, with <10 assets
    "CN_1325960156574879189",
    "CN_8258543408338880789",
    "CN_4206272115616897750",
    "CN_413233497131578182",
    "CP_2845696436723078206",
    "CN_6870637186458717950",
    "CN_1465642096900403277",
    "CN_4903484062166566625",
    "CP_4899418540238054262",
    "CP_1564781859061095794",

]

In [ ]:
tags=[
    "altrisk",
    # "reporting"
    ]


# Define your parameter overrides
runs_configuration = {
    "company_granularity":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": True,
        "apply_retirement":False,
        "apply_decreasing_staggered_shock":False,
    },
    "asset_granularity":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":False,
        "apply_decreasing_staggered_shock":False,
    },
    "asset_granularity_with_retirement":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":True,
        "apply_decreasing_staggered_shock":False,
    },
    "asset_granularity_with_staggered_shock":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":False,
        "apply_decreasing_staggered_shock":True,
    }
}


In [ ]:
run_name


In [ ]:
all_late_sudden_trajectories = {}
all_staggered_shock_results = {}
all_companies_npvs = {}
all_run_params = {}

for run_name, run_params in runs_configuration.items():
    with KedroSession.create(
        project_path=Path.cwd().parent,
        extra_params=run_params,
    ) as session:
        print("================================================")
        print("================================================")
        print(f"Running {run_name}...")
        print("================================================")
        print("================================================")
        session.run(pipeline_name="__default__", tags=tags)

        run_id = uuid.uuid4()

        # late_sudden_trajectories = session.load("late_sudden_trajectories")
        late_sudden_trajectories = pd.read_csv(
            "../data/08_reporting/all_late_sudden_trajectories.csv"
            )
        late_sudden_trajectories["run_id"] = run_id
        staggered_shock_results = pd.read_csv(
            "../data/08_reporting/asset_level_staggered_shock.csv"
            )
        staggered_shock_results["run_id"] = run_id
        companies_npvs = pd.read_csv(
            "../data/07_model_output/company_npv.csv"
        )
        companies_npvs["run_id"] = run_id

        run_params_df = pd.DataFrame([run_params])
        run_params_df["run_id"] = run_id

        all_late_sudden_trajectories[run_name] = late_sudden_trajectories
        all_staggered_shock_results[run_name] = staggered_shock_results
        all_companies_npvs[run_name] = companies_npvs
        all_run_params[run_name] = run_params_df

        # Copy plot folders to workspace/results_v1/{run_name}/
        run_workspace_dir = workspace_dir / run_name
        run_workspace_dir.mkdir(parents=True, exist_ok=True)
        
        # Copy companies_trajectories_plots
        src_trajectories = Path("../data/08_reporting/companies_trajectories_plots")
        dst_trajectories = run_workspace_dir / "companies_trajectories_plots"
        if src_trajectories.exists():
            if dst_trajectories.exists():
                shutil.rmtree(dst_trajectories)
            shutil.copytree(src_trajectories, dst_trajectories)
            print(f"Copied companies_trajectories_plots to {dst_trajectories}")
        
        # Copy companies_staggered_shock_plots  
        src_staggered = Path("../data/08_reporting/companies_staggered_shock_plots")
        dst_staggered = run_workspace_dir / "companies_staggered_shock_plots"
        if src_staggered.exists():
            if dst_staggered.exists():
                shutil.rmtree(dst_staggered)
            shutil.copytree(src_staggered, dst_staggered)
            print(f"Copied companies_staggered_shock_plots to {dst_staggered}")